# M7-v1 fold9 + fold10 scores (the final evaluation prep)

Produces the **frozen M7-v1** scores on fold9 AND fold10 so the final evaluation can load them with **no training**. This is the only M7 training step, isolated on purpose. Config/code are copied **verbatim from `m7_run7.ipynb`** (ResNet1d, strong augmentation, focal loss, early-stop on val loss, 5 seeds, all hyperparameters) — these ARE the frozen M7-v1 scores.

**Discipline:** the 5 models train on **folds 1-8 only** (with an internal early-stop split) and only **predict** fold9 and fold10. Training never sees fold9/fold10. This is inference-producing, not model selection (alpha/threshold/composition are already frozen). **No fold10 metric is computed here** — only raw score arrays are saved; the fold10 metric is the final evaluation's single atomic contact. Single Run All, checkpointed per seed (resume-safe).

## Block 0 — frozen config + verbatim M7-v1 code

In [ ]:
# ===================== SETUP + FROZEN M7-v1 CONFIG (verbatim from m7_run7) =====================
# This notebook is the ONLY M7 training step for the final evaluation: 5 seeds train on folds 1-8, then PREDICT fold9 + fold10.
# Training NEVER sees fold9/fold10. No metric on fold10 here (the final evaluation owns that). No new hyperparameter, no re-tuning.
import os, sys, json, time, warnings, shutil, zipfile
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
ROOT=r'.'
PROC=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
MODELS=os.path.join(ROOT,'models','M7_run7'); CACHE_DIR=os.path.join(PROC,'m7_run2_cache')
os.makedirs(MODELS,exist_ok=True); sys.path.insert(0,SRC)
from sklearn.metrics import average_precision_score, roc_auc_score
# ---- frozen config (EXACT copy from m7_run7) ----
RANDOM_STATE=42; RESO=5000; N_NEG_TRAIN=12000; SEEDS=[0,1,2,3,4]
EPOCHS=60; PATIENCE=10; BATCH=64; LR=1e-3; WD=1e-4; PDROP=0.3
FOCAL_GAMMA=2.0; FOCAL_ALPHA=0.5; VAL_FRAC=0.15; N_JOBS=6
import torch, torch.nn as nn, torch.nn.functional as Fnn
torch.set_num_threads(N_JOBS)
class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)
class ResNet1d(nn.Module):
    def __init__(self,ch=(16,32,64),pdrop=PDROP):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1); self.layer2=BasicBlock1d(ch[0],ch[1],2); self.layer3=BasicBlock1d(ch[1],ch[2],2)
        self.pool=nn.AdaptiveMaxPool1d(1); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(pdrop),nn.Linear(ch[2],1))
    def forward(self,x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x)))))).squeeze(1)
def focal_loss(logits,targets,gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA):
    ce=Fnn.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=torch.where(targets==1,p,1-p)
    a=torch.where(targets==1,torch.full_like(targets,alpha),torch.full_like(targets,1-alpha))
    return (a*(1-pt).pow(gamma)*ce).mean()
def augment(xb):   # strong - verbatim
    T=xb.shape[2]; P=dict(shift=0.08,scale=0.3,noise=0.03,wander=0.08,ldrop=0.45)
    sh=max(1,int(P['shift']*T)); xb=torch.roll(xb,int(torch.randint(-sh,sh+1,(1,)).item()),dims=2)
    xb=xb*((1.0-P['scale'])+2*P['scale']*torch.rand(xb.shape[0],1,1)); xb=xb+P['noise']*torch.randn_like(xb)
    t=torch.linspace(0,1,T).view(1,1,-1); fb=0.5+2.0*torch.rand(xb.shape[0],1,1); ph=2*np.pi*torch.rand(xb.shape[0],1,1)
    xb=xb+P['wander']*torch.sin(2*np.pi*fb*t+ph)
    if torch.rand(1).item()<P['ldrop']: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    if torch.rand(1).item()<0.25: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    return xb
def predict(model,X):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),256):
            out.append(torch.sigmoid(model(torch.tensor(np.ascontiguousarray(X[i:i+256],dtype=np.float32)))).numpy())
    return np.nan_to_num(np.concatenate(out))
def train_one(seed,Xtr,ytr,Xva,yva):
    torch.manual_seed(seed); np.random.seed(seed); rng=np.random.default_rng(seed)
    pos=np.where(ytr==1)[0]; neg=np.where(ytr==0)[0]; half=BATCH//2
    Xv=torch.tensor(np.ascontiguousarray(Xva,dtype=np.float32)); yv=torch.tensor(yva)
    model=ResNet1d(); opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
    steps=max(1,len(ytr)//BATCH); best=1e9; best_state=None; bad=0
    for ep in range(EPOCHS):
        model.train()
        for _ in range(steps):
            bi=np.concatenate([rng.choice(pos,half,True),rng.choice(neg,half,True)])
            xb=augment(torch.tensor(np.ascontiguousarray(Xtr[bi],dtype=np.float32))); yb=torch.tensor(ytr[bi])
            opt.zero_grad(); loss=focal_loss(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=[focal_loss(model(Xv[i:i+256]),yv[i:i+256]).item() for i in range(0,len(Xv),256)]
        vloss=float(np.mean(vl))
        if vloss<best-1e-4: best=vloss; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state); return model
print('Frozen M7-v1 config loaded (verbatim from run7). params:',sum(p.numel() for p in ResNet1d().parameters()))
print('Block 0 OK.')

## Section 1 — signal cache (reuse folds 1-8 & fold9; build fold10)

In [ ]:
# ===================== SECTION 1 — signal cache for folds 1-8 (reuse) + fold9 (reuse) + fold10 (build if needed) =====================
from scipy.signal import butter, sosfiltfilt
from joblib import Parallel, delayed
from tqdm import tqdm
from signal_loading import load_signal, LEADS_CANONICAL
with open(os.path.join(PROC,'filter_config.json')) as f: FCFG=json.load(f)['filter_FINAL']
FS=FCFG['fs']; LEADS=list(LEADS_CANONICAL); LEAD_IDX={L:LEADS_CANONICAL.index(L) for L in LEADS}
SOS=butter(FCFG['order'],[FCFG['low']/(FS/2),FCFG['high']/(FS/2)],btype='band',output='sos')
def bp(x): return sosfiltfilt(SOS,np.asarray(x,dtype=np.float64))
def _zscore(x): return ((x-x.mean(1,keepdims=True))/(x.std(1,keepdims=True)+1e-6)).astype('float16')
# ---- reuse folds 1-8 (mmap) + fold9 from run2 cache ----
NPZ=os.path.join(CACHE_DIR,'m7_run2_data.npz')
def _mmap_member(name):
    out=os.path.join(CACHE_DIR,name+'.npy')
    if not os.path.exists(out):
        with zipfile.ZipFile(NPZ) as zf, zf.open(name+'.npy') as srcf, open(out,'wb') as dst: shutil.copyfileobj(srcf,dst,length=16*1024*1024)
    return np.load(out,mmap_mode='r')
X18=_mmap_member('X18')
z=np.load(NPZ,allow_pickle=True); y18=z['y18']; X9=np.asarray(z['X9']); eid9=np.asarray(z['eid9'],dtype=str); y9=z['y9']; z.close()
meta=pd.read_csv(os.path.join(PROC,'metadata_combined.csv'),dtype={'ecg_id':str})
m9=meta[meta.fold==9]; assert m9.ecg_id.is_unique, 'fold9 ecg_id not unique -> cannot recover source by id'
id2src9=dict(zip(m9.ecg_id,m9.source)); src9=np.array([id2src9.get(str(e),'?') for e in eid9])
print('folds1-8 (mmap) %s (%d WPW) | fold9 %s (%d WPW) | src9 recovered:%s'%(X18.shape,int(y18.sum()),X9.shape,int(y9.sum()),(src9!='?').all()))
# ---- build/load fold10 cache (SAME pipeline as run2) ----
F10=os.path.join(PROC,'m7_fold10_cache.npz')
if os.path.exists(F10):
    z=np.load(F10,allow_pickle=True); X10=z['X10']; eid10=np.asarray(z['eid10'],dtype=str); src10=np.asarray(z['src10'],dtype=str); y10=z['y10']; z.close()
    print('[guard] fold10 cache reloaded')
else:
    d10=meta[meta.fold==10].reset_index(drop=True)
    rows=list(zip(d10.ecg_id.tolist(),d10.source.tolist(),d10.label.values.astype('float32').tolist()))
    def _chunk(rw,outp):
        xs=[];es=[];ss=[];ys=[];fl=0
        for eid,src,lab in rw:
            try:
                raw=load_signal(eid,src); x=np.stack([bp(raw[:,LEAD_IDX[Ld]]) for Ld in LEADS])
                xs.append(_zscore(np.nan_to_num(x))); es.append(eid); ss.append(str(src)); ys.append(lab)
            except Exception: fl+=1
        arr=np.stack(xs).astype('float16') if xs else np.empty((0,12,5000),dtype='float16'); np.save(outp,arr)
        return outp,es,ss,ys,fl
    parts=[p for p in np.array_split(np.arange(len(rows)),N_JOBS) if len(p)]
    paths=[os.path.join(CACHE_DIR,'f10_chunk%d.npy'%j) for j in range(len(parts))]
    res=Parallel(n_jobs=N_JOBS)(delayed(_chunk)([rows[i] for i in part],p) for part,p in zip(parts,paths))
    X10=np.concatenate([np.load(pth) for pth,_,_,_,_ in res],axis=0)
    eid10=np.array([e for _,es,_,_,_ in res for e in es],dtype='U16'); src10=np.array([s for _,_,ss,_,_ in res for s in ss],dtype='U16')
    y10=np.array([l for _,_,_,ys,_ in res for l in ys],dtype='float32'); nf=sum(r[4] for r in res)
    for pth,_,_,_,_ in res:
        if os.path.exists(pth): os.remove(pth)
    np.savez(F10,X10=X10,eid10=eid10,src10=src10,y10=y10)
    print('[guard] fold10 cache written | %d skipped (malformed header)'%nf)
print('fold10 %s | %d WPW (expect 14) | %d records'%(X10.shape,int(y10.sum()),len(X10)))
print('Section 1 OK.')

## Section 2 — train 5 seeds on folds 1-8, predict fold9 + fold10 (checkpointed)

In [ ]:
# ===================== SECTION 2 — train 5 seeds on folds 1-8, predict fold9 + fold10 (checkpointed) =====================
# Training pool = ALL folds 1-8 (115 WPW + N_NEG_TRAIN subsampled negatives). Internal stratified val split for early-stop.
# Fixed pool/val split shared across seeds (as in run7). Predict X9 and X10 after each seed. Per-seed checkpoint = resume-safe.
rng=np.random.default_rng(2025)
pos=np.where(y18==1)[0]; negall=np.where(y18==0)[0]
neg=rng.choice(negall,min(N_NEG_TRAIN,len(negall)),replace=False)
vp=pos.copy(); vn=neg.copy(); rng.shuffle(vp); rng.shuffle(vn)
nvp=max(6,int(VAL_FRAC*len(vp))); nvn=int(VAL_FRAC*len(vn))
val_idx=np.concatenate([vp[:nvp],vn[:nvn]]); pool=np.concatenate([pos,neg]); tr_idx=np.setdiff1d(pool,val_idx)
Xtr=np.ascontiguousarray(X18[tr_idx],dtype=np.float32); ytr=y18[tr_idx]; Xva=X18[val_idx]; yva=y18[val_idx]
print('train pool %d (%d WPW) | val %d (%d WPW)'%(len(tr_idx),int(ytr.sum()),len(val_idx),int(yva.sum())))
CK=os.path.join(MODELS,'m7_f9f10_ckpt.npz')
if os.path.exists(CK):
    c=np.load(CK); seed9=[r for r in c['p9']]; seed10=[r for r in c['p10']]; done=int(c['done']); print('[ckpt] resumed at seed %d/%d'%(done,len(SEEDS)))
else:
    seed9=[]; seed10=[]; done=0
t0=time.time()
for si,s in enumerate(SEEDS):
    if si<done: continue
    ts=time.time(); model=train_one(s,Xtr,ytr,Xva,yva)
    seed9.append(predict(model,X9)); seed10.append(predict(model,X10))
    np.savez(CK,p9=np.array(seed9),p10=np.array(seed10),done=si+1)
    print('  seed %d done | %.1f min | fold9 AP(this seed) %.3f'%(s,(time.time()-ts)/60,average_precision_score(y9,seed9[-1])))
f9=np.mean(seed9,0).astype('float32'); f10=np.mean(seed10,0).astype('float32')
print('Section 2 OK | %d seeds | total %.1f min'%(len(seed9),(time.time()-t0)/60))

## Section 3 — save score artifacts

In [ ]:
# ===================== SECTION 3 — save score artifacts (fold9 + fold10) =====================
pd.DataFrame({'ecg_id':np.asarray(eid9,dtype=str),'source':np.asarray(src9,dtype=str),'label':y9,'m7v1_score':f9}
    ).to_csv(os.path.join(MODELS,'m7v1_fold9_scores.csv'),index=False)
pd.DataFrame({'ecg_id':np.asarray(eid10,dtype=str),'source':np.asarray(src10,dtype=str),'label':y10,'m7v1_score':f10}
    ).to_csv(os.path.join(MODELS,'m7v1_fold10_scores.csv'),index=False)
print('saved: m7v1_fold9_scores.csv (%d rows) | m7v1_fold10_scores.csv (%d rows)'%(len(f9),len(f10)))
# sanity: reproduce vs existing run7 fold9 scores (5 models on 8 folds here vs 40 models on 7 folds in run7 -> high corr expected)
old_p=os.path.join(MODELS,'m7v1_fold9_scores.npy')
if os.path.exists(old_p):
    old=np.load(old_p)
    if len(old)==len(f9):
        corr=float(np.corrcoef(old,f9)[0,1]); mad=float(np.max(np.abs(old-f9)))
        print('fold9 reproduction vs run7 m7v1_fold9_scores.npy : corr=%.4f | max|diff|=%.4f  %s'%(corr,mad,'OK (reproduces frozen M7-v1)' if corr>=0.95 else '*** LOW CORR - FLAG ***'))
    else: print('fold9 length mismatch vs existing npy (%d vs %d) - cannot compare'%(len(old),len(f9)))
else: print('no existing m7v1_fold9_scores.npy to compare')
print('Section 3 OK.')

## Section 4 — sanity only (NOT a fold10 evaluation)

In [ ]:
# ===================== SECTION 4 — sanity ONLY (NOT a fold10 evaluation) =====================
# fold9 = validation (used throughout the project) -> AP/AUC are fine to look at and confirm M7-v1 reproduces ~0.83.
ap9=float(average_precision_score(y9,f9)); auc9=float(roc_auc_score(y9,f9))
print('fold9 (validation, %d WPW): AP=%.3f  AUC=%.3f   (run7 reported ~0.834)'%(int(y9.sum()),ap9,auc9))
# fold10 = RESERVED. Print ONLY the score distribution -> NO AP/AUC/confusion here.
q=np.percentile(f10,[0,25,50,75,100])
print('\nfold10 score distribution ONLY (NO metric here - fold10 metric is the final evaluation single atomic contact):')
print('  min/Q1/median/Q3/max = %.3f / %.3f / %.3f / %.3f / %.3f  (n=%d)'%(q[0],q[1],q[2],q[3],q[4],len(f10)))
wpw10=f10[y10==1]; neg10=f10[y10==0]
# express WPW scores as percentile within the fold10 negatives (descriptive, not a metric)
pcts=[round(float((neg10<=w).mean()),3) for w in np.sort(wpw10)[::-1]]
print('  %d fold10 WPW scores (sorted desc): %s'%(int(y10.sum()),np.round(np.sort(wpw10)[::-1],3).tolist()))
print('  their percentile within fold10 negatives: %s'%pcts)
print('  negatives score median=%.3f'%float(np.median(neg10)))
print('\n>>> fold10 AP/AUC/confusion are NOT computed here. They belong to the final evaluation (single atomic test-set contact).')
print('Section 4 OK.')